# Nettoyage des données DVF - Ille-et-Vilaine (2021-2025)

Ce notebook charge les fichiers DVF bruts (un par année), les assemble, filtre les colonnes et les types de biens pertinents, traite les valeurs manquantes et aberrantes, calcule le prix au m², puis exporte un fichier propre dans `data/cleaned/`.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Matplotlib is building the font cache; this may take a moment.


In [7]:
# Liste des années couvertes par l'analyse
annees = [2021, 2022, 2023, 2024, 2025]

# Liste vide qui va accueillir un DataFrame par année
dataframes = []

for annee in annees:
    chemin = f"../data/raw/dvf_35_{annee}.csv.gz"
    df_annee = pd.read_csv(chemin, low_memory=False)
    df_annee["annee"] = annee
    dataframes.append(df_annee)

df = pd.concat(dataframes, ignore_index=True)

print(df.shape)
df.head()

(337019, 41)


,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,annee
0,2021-591508,2021-01-07,1,Vente,160000.0,7.0,NaN,RUE BEAUGEARD LANCELOT,0750,35700.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,-1.656551,48.119053,2021
1,2021-591508,2021-01-07,1,Vente,160000.0,7.0,NaN,RUE BEAUGEARD LANCELOT,0750,35700.0,...,55.0,3.0,NaN,NaN,NaN,NaN,NaN,-1.656551,48.119053,2021
2,2021-591509,2021-01-06,1,Vente,225000.0,6.0,NaN,BD DES METAIRIES,0432,35510.0,...,81.0,4.0,NaN,NaN,NaN,NaN,NaN,-1.599096,48.121689,2021
3,2021-591510,2021-01-07,1,Vente,304000.0,12.0,NaN,ALL EMILE GERNIGON,0963,35136.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,-1.699642,48.091458,2021
4,2021-591510,2021-01-07,1,Vente,304000.0,12.0,NaN,ALL EMILE GERNIGON,0963,35136.0,...,96.0,4.0,NaN,NaN,NaN,NaN,NaN,-1.699642,48.091458,2021


## Exploration initiale

Avant de nettoyer, on regarde d'abord ce que contient réellement le tableau : les types de colonnes, les valeurs manquantes, et les doublons potentiels.

In [ ]:
# J'affiche la liste des colonnes de mon tableau, leur nom, le nombre 
# de valeurs non vide qu'elles contiennent et leur type

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 337019 entries, 0 to 337018
Data columns (total 41 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id_mutation                   337019 non-null  str    
 1   date_mutation                 337019 non-null  str    
 2   numero_disposition            337019 non-null  int64  
 3   nature_mutation               337019 non-null  str    
 4   valeur_fonciere               333524 non-null  float64
 5   adresse_numero                216238 non-null  float64
 6   adresse_suffixe               13028 non-null   str    
 7   adresse_nom_voie              332070 non-null  str    
 8   adresse_code_voie             332121 non-null  str    
 9   code_postal                   332118 non-null  float64
 10  code_commune                  337019 non-null  int64  
 11  nom_commune                   337019 non-null  str    
 12  code_departement              337019 non-null  int64  


In [10]:
colonnes_utiles = [
    "date_mutation", "annee", "nature_mutation", "valeur_fonciere",
    "surface_reelle_bati", "type_local", "code_commune", "nom_commune",
    "longitude", "latitude"
]

df[colonnes_utiles].isnull().sum()

date_mutation               0
annee                       0
nature_mutation             0
valeur_fonciere          3495
surface_reelle_bati    232562
type_local             157570
code_commune                0
nom_commune                 0
longitude                8494
latitude                 8494
dtype: int64

In [11]:
df[df["type_local"].isnull()][["nature_mutation", "code_nature_culture", "nature_culture"]].head(10)

,nature_mutation,code_nature_culture,nature_culture
8,Vente en l'état futur d'achèvement,NaN,NaN
9,Vente en l'état futur d'achèvement,NaN,NaN
10,Vente en l'état futur d'achèvement,NaN,NaN
17,Vente,S,sols
21,Vente en l'état futur d'achèvement,NaN,NaN
22,Vente en l'état futur d'achèvement,NaN,NaN
37,Vente,S,sols
39,Vente en l'état futur d'achèvement,NaN,NaN
40,Vente en l'état futur d'achèvement,NaN,NaN
41,Vente,S,sols
